# AI@UCI — Week 3: Intro to ML & Basic Classifiers
## From Intuition to Code

**Pipeline:** data → features + labels → feature space → train/test split → KNN → prediction → evaluation

We use the same Cat vs Dog story from the slides. The core activity is designed for ~15 minutes; the last visualization is optional.

In [ ]:
# %pip install -q numpy pandas matplotlib scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
rng = np.random.default_rng(7)

## 1. Examples become data
Each row is one **sample**. `ear_size` and `fur_length` are **features**. `label` is the answer we want to predict.

In [ ]:
n = 40
cats = pd.DataFrame({'ear_size': rng.normal(6.4,1.0,n), 'fur_length': rng.normal(6.0,1.0,n), 'label':'cat'})
dogs = pd.DataFrame({'ear_size': rng.normal(4.5,1.0,n), 'fur_length': rng.normal(4.3,1.0,n), 'label':'dog'})
df = pd.concat([cats,dogs], ignore_index=True)
noise_idx = rng.choice(len(df), size=6, replace=False)
df.loc[noise_idx,'label'] = df.loc[noise_idx,'label'].map({'cat':'dog','dog':'cat'})
df.sample(8, random_state=3)

## 2. Where does the data live?
When each feature becomes an axis, every sample becomes a point in **feature space**.

In [ ]:
for label, group in df.groupby('label'):
    plt.scatter(group['ear_size'], group['fur_length'], label=label, alpha=0.8)
plt.xlabel('ear_size'); plt.ylabel('fur_length'); plt.title('Cat vs Dog Feature Space'); plt.legend(); plt.show()

In [ ]:
X = df[['ear_size','fur_length']]
y = df['label']
print('X shape:', X.shape, '| y shape:', y.shape)

## 3. Split before you learn
The training set is available to the model. The test set stays hidden so we can check **generalization**.

In [ ]:
# TODO 1: 25% test data, random_state=42, stratify=y
X_train, X_test, y_train, y_test = ...
print('Train:', len(X_train), '| Test:', len(X_test))

## 4. KNN intuition — do one prediction by hand
For a new point, KNN measures distance, keeps the `k` closest labeled examples, then lets them vote.

In [ ]:
new_point = np.array([6.2,5.8])
distances = np.sqrt(np.sum((X_train.to_numpy() - new_point)**2, axis=1))
distance_table = X_train.copy()
distance_table['label'] = y_train.to_numpy()
distance_table['distance'] = distances
distance_table.sort_values('distance').head(8)

In [ ]:
# TODO 2: keep the 5 nearest rows, count labels, choose the majority label
nearest_5 = ...
votes = ...
manual_prediction = ...
print(votes)
print('Manual prediction:', manual_prediction)

## 5. Let scikit-learn do the same thing
This `k=5` example is completed first. Then you will repeat the pattern for other values of `k`.

In [ ]:
knn_5 = KNeighborsClassifier(n_neighbors=5)
knn_5.fit(X_train, y_train)
pred_5 = knn_5.predict(X_test)
train_acc_5 = accuracy_score(y_train, knn_5.predict(X_train))
test_acc_5 = accuracy_score(y_test, pred_5)
print(f'k=5 training accuracy: {train_acc_5:.3f}')
print(f'k=5 test accuracy:     {test_acc_5:.3f}')

## 6. What does k change?
Compare a very local classifier (`k=1`) with a much broader one (`k=25`).

In [ ]:
# TODO 3: build, fit, and evaluate k=1
knn_1 = ...
...
train_acc_1 = ...
test_acc_1 = ...
print(f'k=1 training accuracy: {train_acc_1:.3f}')
print(f'k=1 test accuracy:     {test_acc_1:.3f}')

In [ ]:
# TODO 4: build, fit, and evaluate k=25
knn_25 = ...
...
train_acc_25 = ...
test_acc_25 = ...
print(f'k=25 training accuracy: {train_acc_25:.3f}')
print(f'k=25 test accuracy:     {test_acc_25:.3f}')

In [ ]:
comparison = pd.DataFrame({'k':[1,5,25], 'training_accuracy':[train_acc_1,train_acc_5,train_acc_25], 'test_accuracy':[test_acc_1,test_acc_5,test_acc_25]})
comparison

### Discuss
Why can `k=1` look perfect on the training set without being the most trustworthy model on unseen data? This is the bridge from **memorization** to **generalization / overfitting**.

In [ ]:
k_values=[1,3,5,7,9,15,25]; train_scores=[]; test_scores=[]
for k in k_values:
    model=KNeighborsClassifier(n_neighbors=k).fit(X_train,y_train)
    train_scores.append(accuracy_score(y_train,model.predict(X_train)))
    test_scores.append(accuracy_score(y_test,model.predict(X_test)))
plt.plot(k_values,train_scores,marker='o',label='training accuracy')
plt.plot(k_values,test_scores,marker='o',label='test accuracy')
plt.xlabel('k'); plt.ylabel('accuracy'); plt.title('Training vs Test Accuracy'); plt.ylim(0.6,1.02); plt.legend(); plt.show()

## 7. Predict a brand-new point

In [ ]:
# TODO 5: make a one-row DataFrame for ear_size=6.2, fur_length=5.8, then predict with knn_5
new_sample = ...
prediction = ...
print('Prediction:', prediction[0])

## Wrap-up
You implemented: **data → features + labels → feature space → train/test split → KNN → prediction → accuracy**

> The goal is not to memorize the training data. The goal is to generalize.

---
## Optional: visualize KNN decision regions
Run this only if time remains. It previews the idea that different classifiers carve up the same feature space in different ways.

In [ ]:
def plot_knn_regions(model,X_data,y_data,title):
    x0,x1=X_data['ear_size'].min()-1,X_data['ear_size'].max()+1
    y0,y1=X_data['fur_length'].min()-1,X_data['fur_length'].max()+1
    xx,yy=np.meshgrid(np.linspace(x0,x1,220),np.linspace(y0,y1,220))
    grid=pd.DataFrame({'ear_size':xx.ravel(),'fur_length':yy.ravel()})
    z=pd.Series(model.predict(grid)).map({'dog':0,'cat':1}).to_numpy().reshape(xx.shape)
    plt.contourf(xx,yy,z,alpha=0.15,levels=[-0.5,0.5,1.5])
    for label,g in pd.concat([X_data,y_data],axis=1).groupby('label'):
        plt.scatter(g['ear_size'],g['fur_length'],label=label,alpha=0.8)
    plt.xlabel('ear_size'); plt.ylabel('fur_length'); plt.title(title); plt.legend(); plt.show()
plot_knn_regions(knn_5,X_train,y_train,'KNN Decision Regions (k=5)')